# 訓練實驗（PC / Colab，有 GPU）

先跑第 0 節安裝環境。之後每一格程式碼 = 一個訓練實驗：上面寫設定，最後一行 `run(...)` 會呼叫 `train.py` 訓練，訓練完印出 **test** 的結果。

- **不會重複訓練**：檔名由所有設定自動產生（例如 `resnet18_aug2_e10_lr0.001_bs16_seed0`）。`models/` 裡已經有同名的模型時就不訓練，直接讀出上次的 test 結果。要重新訓練就把那個模型的 `.pth` / `.json` 刪掉。
- 設定沒寫的（`epochs = None`、`lr = None`）用該網路檔的 `DEFAULTS`（目前是 10 epochs、lr 1e-3）；batch size 預設 16。
- `augment = True`：每張訓練圖用 7 種方法（翻轉、旋轉、裁切、顏色、噪聲、模糊、壓縮）裡最多 2 種的所有組合（29 種，含原圖）展開，一個 epoch 看 277 × 29 張；`False` 只用原圖。
- `seeds = [0]` 只跑一次；要看平均和標準差就改成 `[17, 29, 43]`（和隊友相同），檔案會變成 `..._s17_base.pth` 等，另外寫一份 `..._seeds.json`。
- 最後一格把這次執行過的實驗（含跳過的）整理成一張表。
- 想加實驗：複製一格、改設定即可。

## 0. 安裝環境

在 `project4/` 資料夾裡開這個 notebook。下面這格會用目前 kernel 的 Python 安裝 `requirements-pc.txt`（torch、torchvision、torch-pruning、onnx、onnxruntime、numpy、pillow、opencv-python-headless）；已經裝好的套件會直接略過，所以每次 Run All 都可以留著。

- Colab 已經內建有 CUDA 的 torch，不會重裝。
- 自己的電腦要用 GPU 的話，torch 請先照 https://pytorch.org 選對 CUDA 版本安裝，再跑這格。
- 資料集放在 `data/splits/{train,validation,test}/`（隊友切好的資料夾）。

In [1]:
import subprocess
import sys

# Some virtual environments (e.g. made by uv) come without pip: add it first
if subprocess.call([sys.executable, "-m", "pip", "--version"], stdout=subprocess.DEVNULL) != 0:
    # Bundled pip installer of the standard library
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
# Install / check the PC packages with this kernel's Python (already installed ones are skipped)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-pc.txt"])
import torch
# Show what will be used for training
print("torch %s | CUDA available: %s%s" % (torch.__version__, torch.cuda.is_available(),
      " (%s)" % torch.cuda.get_device_name(0) if torch.cuda.is_available() else " -> training on CPU is very slow"))

torch 2.15.0+cu132 | CUDA available: True (NVIDIA GeForce GTX 1660 Ti)


In [2]:
# --- Shared settings (change here once for every experiment) -----------------
import json
import os
import subprocess
import sys

# Network registry (for the default epochs / lr and the file-name prefix)
from loader import ARCHS
# Augmentation switch -> file-name tag ("aug2" / "none")
from augmentation import Augmenter

# Dataset with the team's train/ validation/ test/ folders (test is needed for the results)
DATA_DIR = "data/splits"
# Where train.py writes the models (.pth + .json); live_demo / convert_trt read from here
OUT_DIR = "models"
# DataLoader worker processes (0 if the machine complains about shared memory)
WORKERS = 4
# Results of every experiment run in this session, for the summary table
RESULTS = {}


# File-name stem built from every setting, so different settings never share a name
def run_name(arch, epochs, lr, augment, batch_size, seeds):
    """e.g. 'resnet18_aug_e10_lr0.001_bs16_seed0' (several seeds: no seed part, train.py adds _s<seed>)."""
    # Augmentation part of the name, same tag train.py uses
    aug = Augmenter(enabled=augment).tag()
    # Network, augmentation, epochs, learning rate and batch size
    name = "%s_%s_e%d_lr%g_bs%d" % (ARCHS[arch].FILE_TAG, aug, epochs, lr, batch_size)
    # A single seed is part of the name; several seeds get their own suffixes from train.py
    return name + ("_seed%d" % seeds[0] if len(seeds) == 1 else "")


# Test metrics of a finished experiment, read from the files train.py wrote
def read_results(name, seeds):
    """{"test": {...}, "val": {...}, "macs": ...}; several seeds give mean / std per metric."""
    # One seed: the baseline's .json
    if len(seeds) == 1:
        # Metadata written next to the .pth
        with open(os.path.join(OUT_DIR, name + "_base.json")) as f:
            # Parsed metadata
            meta = json.load(f)
        # Test (None without a test split), validation and size
        return {"test": meta.get("test_metrics"), "val": meta.get("val_metrics"), "macs": meta.get("macs")}
    # Several seeds: the summary train.py wrote
    with open(os.path.join(OUT_DIR, name + "_seeds.json")) as f:
        # Baseline statistics
        base = json.load(f)["summary"]["base"]
    # Mean / std per metric for test and validation
    return {"test": base.get("test"), "val": base.get("val"), "macs": None}


# Has this exact experiment been trained already?
def finished(name, seeds):
    # One seed: the baseline weights and their .json; several seeds: the summary file
    files = [name + "_base.pth", name + "_base.json"] if len(seeds) == 1 else [name + "_seeds.json"]
    # All of them must exist
    return all(os.path.exists(os.path.join(OUT_DIR, f)) for f in files)


# Print the test (or, without a test split, validation) metrics
def show(name, res, seeds):
    # Prefer test; fall back to validation and say so
    split, m = ("test", res["test"]) if res["test"] else ("val (no test split!)", res["val"])
    # One seed: plain numbers
    if len(seeds) == 1:
        # MSE and pixel errors
        print("%s  [%s]  mse %.5f | mean %.1f px | x %.1f px | y %.1f px | p95 %.1f px"
              % (name, split, m["mse"], m["mean_px"], m["mae_x_px"], m["mae_y_px"], m["p95_px"]))
    # Several seeds: mean +- std
    else:
        # MSE and pixel errors with their spread over the seeds
        print("%s  [%s, seeds %s]  mse %.5f +- %.5f | mean %.1f +- %.1f px | p95 %.1f +- %.1f px"
              % (name, split, seeds, m["mse"]["mean"], m["mse"]["std"], m["mean_px"]["mean"],
                 m["mean_px"]["std"], m["p95_px"]["mean"], m["p95_px"]["std"]))


# Train one experiment unless it already exists, then print its test results
def run(arch, epochs=None, lr=None, augment=False, batch_size=16, seeds=(0,)):
    # Defaults of the chosen network for anything left as None
    epochs = epochs if epochs is not None else ARCHS[arch].DEFAULTS["epochs"]
    # Learning rate default
    lr = lr if lr is not None else ARCHS[arch].DEFAULTS["lr"]
    # Seeds as a list
    seeds = list(seeds)
    # File-name stem of this experiment
    name = run_name(arch, epochs, lr, augment, batch_size, seeds)
    # Same settings trained before: reuse the result
    if finished(name, seeds):
        # Tell the user why nothing is trained
        print("already trained, skipping: %s (delete its files in %s/ to train again)" % (name, OUT_DIR))
    # New experiment: run train.py
    else:
        # Command line, same as typing it in a terminal
        cmd = [sys.executable, "train.py", "--arch", arch, "--data", DATA_DIR, "--out", OUT_DIR,
               "--name", name, "--epochs", str(epochs), "--lr", str(lr), "--batch-size", str(batch_size),
               "--workers", str(WORKERS), "--seeds"] + [str(s) for s in seeds]
        # Augmentation switch
        cmd += ["--augment"] if augment else []
        # Show the command
        print("$ " + " ".join(cmd), flush=True)
        # Run it and stream its log into the notebook
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True,
                                env=dict(os.environ, PYTHONUNBUFFERED="1"))
        # Every log line as it comes
        for line in proc.stdout:
            # Pass it through
            print(line, end="", flush=True)
        # A failed run must not look finished
        if proc.wait() != 0:
            # Stop Run All here so the error is visible
            raise RuntimeError("train.py failed for %s (exit code %d)" % (name, proc.returncode))
    # Results from the files
    res = read_results(name, seeds)
    # Remember them for the summary
    RESULTS[name] = dict(res, arch=arch, seeds=seeds)
    # Print the test results
    show(name, res, seeds)

## 1. 五種網路，augmentation 開，10 epochs

In [3]:
# Network
model = "resnet18"
# Training length
epochs = 70
# Learning rate (None = the network's DEFAULTS)
lr = None
# Augmentation on / off (subsets of at most 2 of the 7 methods)
augment = True
# Batch size
batch_size = 16
# Training seeds ([17, 29, 43] for mean +- std)
seeds = [0]
# Train (or reuse) and print the test results
run(model, epochs, lr, augment, batch_size, seeds)

$ /home/user/jupyter-env/bin/python3 train.py --arch resnet18 --data data/splits --out models --name resnet18_aug2_e70_lr0.001_bs16_seed0 --epochs 70 --lr 0.001 --batch-size 16 --workers 4 --seeds 0 --augment
data: 277 train (8033 samples per epoch) / 60 val / 61 test, ResNet18, Augmenter(enabled=True, 29 variants per image), lane marking on, device cuda
split: {'split': 'folders', 'train': 277, 'validation': 60, 'test': 61, 'skipped_unreadable': 0}
training: 70 epochs lr 0.001, finetune 15 epochs lr 0.0001, seeds [0]
[base] epoch   1/70  train 0.05717  val 0.02728 (23.2 px)  (42.2s) *
[base] epoch   2/70  train 0.02578  val 0.02121 (20.7 px)  (50.0s) *
[base] epoch   3/70  train 0.01633  val 0.06434 (35.8 px)  (57.1s)
[base] epoch   4/70  train 0.01559  val 0.03020 (24.4 px)  (60.9s)
[base] epoch   5/70  train 0.01143  val 0.03525 (26.3 px)  (64.3s)
[base] epoch   6/70  train 0.01007  val 0.04449 (28.3 px)  (65.7s)
[base] epoch   7/70  train 0.00921  val 0.02788 (22.9 px)  (68.1s)
[ba

In [4]:
# Network
model = "mobilenet_v2"
# Training length
epochs = 70
# Learning rate (None = the network's DEFAULTS)
lr = None
# Augmentation on / off (subsets of at most 2 of the 7 methods)
augment = True
# Batch size
batch_size = 16
# Training seeds ([17, 29, 43] for mean +- std)
seeds = [0]
# Train (or reuse) and print the test results
run(model, epochs, lr, augment, batch_size, seeds)

$ /home/user/jupyter-env/bin/python3 train.py --arch mobilenet_v2 --data data/splits --out models --name mobilenetv2_aug2_e70_lr0.001_bs16_seed0 --epochs 70 --lr 0.001 --batch-size 16 --workers 4 --seeds 0 --augment
data: 277 train (8033 samples per epoch) / 60 val / 61 test, MobileNetV2, Augmenter(enabled=True, 29 variants per image), lane marking on, device cuda
split: {'split': 'folders', 'train': 277, 'validation': 60, 'test': 61, 'skipped_unreadable': 0}
training: 70 epochs lr 0.001, finetune 15 epochs lr 0.0001, seeds [0]
[base] epoch   1/70  train 0.05392  val 0.02928 (23.6 px)  (86.8s) *
[base] epoch   2/70  train 0.01974  val 0.03082 (24.7 px)  (87.8s)
[base] epoch   3/70  train 0.01507  val 0.07411 (38.4 px)  (89.9s)
[base] epoch   4/70  train 0.03519  val 0.03453 (26.2 px)  (88.8s)
[base] epoch   5/70  train 0.02271  val 0.03967 (27.0 px)  (89.2s)
[base] epoch   6/70  train 0.01726  val 0.05175 (30.7 px)  (89.9s)
[base] epoch   7/70  train 0.04186  val 0.04213 (27.4 px)  (88

In [5]:
# Network
model = "resnet18_coord"
# Training length
epochs = 70
# Learning rate (None = the network's DEFAULTS)
lr = None
# Augmentation on / off (subsets of at most 2 of the 7 methods)
augment = True
# Batch size
batch_size = 16
# Training seeds ([17, 29, 43] for mean +- std)
seeds = [0]
# Train (or reuse) and print the test results
run(model, epochs, lr, augment, batch_size, seeds)

$ /home/user/jupyter-env/bin/python3 train.py --arch resnet18_coord --data data/splits --out models --name resnet18coord_aug2_e70_lr0.001_bs16_seed0 --epochs 70 --lr 0.001 --batch-size 16 --workers 4 --seeds 0 --augment
data: 277 train (8033 samples per epoch) / 60 val / 61 test, ResNet18 CoordConv, Augmenter(enabled=True, 29 variants per image), lane marking on, device cuda
split: {'split': 'folders', 'train': 277, 'validation': 60, 'test': 61, 'skipped_unreadable': 0}
training: 70 epochs lr 0.001, finetune 15 epochs lr 0.0001, seeds [0]
[base] epoch   1/70  train 0.05898  val 0.03106 (24.2 px)  (78.8s) *
[base] epoch   2/70  train 0.02754  val 0.02862 (22.7 px)  (79.8s) *
[base] epoch   3/70  train 0.01853  val 0.10296 (46.7 px)  (80.0s)
[base] epoch   4/70  train 0.01379  val 0.02891 (22.6 px)  (79.3s)
[base] epoch   5/70  train 0.01258  val 0.06890 (34.5 px)  (80.1s)
[base] epoch   6/70  train 0.01238  val 0.05447 (31.3 px)  (79.5s)
[base] epoch   7/70  train 0.00932  val 0.03149 (

In [6]:
# Network
model = "resnet18_dsnt"
# Training length
epochs = 70
# Learning rate (None = the network's DEFAULTS)
lr = None
# Augmentation on / off (subsets of at most 2 of the 7 methods)
augment = True
# Batch size
batch_size = 16
# Training seeds ([17, 29, 43] for mean +- std)
seeds = [0]
# Train (or reuse) and print the test results
run(model, epochs, lr, augment, batch_size, seeds)

$ /home/user/jupyter-env/bin/python3 train.py --arch resnet18_dsnt --data data/splits --out models --name resnet18dsnt_aug2_e70_lr0.001_bs16_seed0 --epochs 70 --lr 0.001 --batch-size 16 --workers 4 --seeds 0 --augment
data: 277 train (8033 samples per epoch) / 60 val / 61 test, ResNet18 DSNT, Augmenter(enabled=True, 29 variants per image), lane marking on, device cuda
split: {'split': 'folders', 'train': 277, 'validation': 60, 'test': 61, 'skipped_unreadable': 0}
training: 70 epochs lr 0.001, finetune 15 epochs lr 0.0001, seeds [0]
[base] epoch   1/70  train 0.05833  val 0.03060 (23.9 px)  (79.1s) *
[base] epoch   2/70  train 0.04543  val 0.02770 (21.7 px)  (78.6s) *
[base] epoch   3/70  train 0.03785  val 0.03763 (26.1 px)  (80.3s)
[base] epoch   4/70  train 0.04129  val 0.03406 (24.3 px)  (79.3s)
[base] epoch   5/70  train 0.03355  val 0.02398 (20.0 px)  (79.9s) *
[base] epoch   6/70  train 0.02821  val 0.04639 (26.7 px)  (79.8s)
[base] epoch   7/70  train 0.02293  val 0.05359 (29.7 

In [7]:
# Network
model = "resnet18_coord_dsnt"
# Training length
epochs = 70
# Learning rate (None = the network's DEFAULTS)
lr = None
# Augmentation on / off (subsets of at most 2 of the 7 methods)
augment = True
# Batch size
batch_size = 16
# Training seeds ([17, 29, 43] for mean +- std)
seeds = [0]
# Train (or reuse) and print the test results
run(model, epochs, lr, augment, batch_size, seeds)

$ /home/user/jupyter-env/bin/python3 train.py --arch resnet18_coord_dsnt --data data/splits --out models --name resnet18coorddsnt_aug2_e70_lr0.001_bs16_seed0 --epochs 70 --lr 0.001 --batch-size 16 --workers 4 --seeds 0 --augment
data: 277 train (8033 samples per epoch) / 60 val / 61 test, ResNet18 CoordConv+DSNT, Augmenter(enabled=True, 29 variants per image), lane marking on, device cuda
split: {'split': 'folders', 'train': 277, 'validation': 60, 'test': 61, 'skipped_unreadable': 0}
training: 70 epochs lr 0.001, finetune 15 epochs lr 0.0001, seeds [0]
[base] epoch   1/70  train 0.05805  val 0.02602 (21.4 px)  (77.9s) *
[base] epoch   2/70  train 0.04302  val 0.03295 (23.6 px)  (78.8s)
[base] epoch   3/70  train 0.03762  val 0.05697 (31.7 px)  (78.7s)
[base] epoch   4/70  train 0.03559  val 0.03067 (21.6 px)  (77.9s)
[base] epoch   5/70  train 0.02758  val 0.03909 (26.2 px)  (79.0s)
[base] epoch   6/70  train 0.02667  val 0.04449 (27.0 px)  (78.9s)
[base] epoch   7/70  train 0.02235  v

## 2. 總表（這次執行過的所有實驗，test 結果）

In [8]:
# One row per experiment: test metrics (mean over seeds when there are several)
def value(m, key):
    # Several seeds store {"mean", "std", "values"}; one seed stores the number
    return m[key]["mean"] if isinstance(m[key], dict) else m[key]


# Header of the table
print("%-46s %-8s %9s %9s %8s %8s %8s %7s" % ("experiment", "split", "mse", "mean px", "x px", "y px",
                                              "p95 px", "GMACs"))
# Experiments sorted by mean pixel error (best first)
rows = sorted(RESULTS.items(), key=lambda kv: value(kv[1]["test"] or kv[1]["val"], "mean_px"))
# One line each
for name, res in rows:
    # Test metrics, or validation when there is no test split
    split, m = ("test", res["test"]) if res["test"] else ("val", res["val"])
    # Size in GMACs when known
    macs = "%.2f" % (res["macs"] / 1e9) if res["macs"] else "-"
    # Seed count marker
    tag = name + (" (%d seeds)" % len(res["seeds"]) if len(res["seeds"]) > 1 else "")
    # The row
    print("%-46s %-8s %9.5f %9.1f %8.1f %8.1f %8.1f %7s" % (tag, split, value(m, "mse"), value(m, "mean_px"),
                                                         value(m, "mae_x_px"), value(m, "mae_y_px"),
                                                         value(m, "p95_px"), macs))

experiment                                     split          mse   mean px     x px     y px   p95 px   GMACs
resnet18coorddsnt_aug2_e70_lr0.001_bs16_seed0  test       0.03287      21.0      9.5     16.4     61.0    3.14
resnet18dsnt_aug2_e70_lr0.001_bs16_seed0       test       0.02717      22.5     10.0     17.6     48.1    3.06
resnet18_aug2_e70_lr0.001_bs16_seed0           test       0.02595      22.9     11.8     16.7     45.1    1.82
resnet18coord_aug2_e70_lr0.001_bs16_seed0      test       0.03174      24.4      8.9     20.6     51.7    1.90
mobilenetv2_aug2_e70_lr0.001_bs16_seed0        test       0.02950      24.8     10.0     19.8     44.3    0.32
